## 1. Dataset Structure Check

In [51]:
import os

def count_images(path):
    return len([f for f in os.listdir(path) if f.endswith((".jpg", ".png", ".jpeg"))])

print("Dataset overview loaded from YAML:\n")
print(open("dataset.yaml").read())
print("")

train = count_images("dataset/train/images")
val = count_images("dataset/val/images")
test = count_images("dataset/test/images")

print(f"Train: {train}")
print(f"Val: {val}")
print(f"Test: {test}")

Dataset overview loaded from YAML:

train: dataset/train/images
val: dataset/val/images
test: dataset/test/images

nc: 3

names:
  0: crack
  1: potholes
  2: wall_peeling

Train: 546
Val: 117
Test: 117


In [52]:
from ultralytics import YOLO
model = YOLO("yolo26s.pt")
model.info()
print(model.model)

YOLO26s summary: 260 layers, 10,009,784 parameters, 0 gradients, 22.8 GFLOPs
DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C3k2(
      (cv1): Conv(
        (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(96, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affin

## 2. IoU + Evaluation Function

In [53]:
from ultralytics import YOLO
import numpy as np
import cv2
import glob
import os

def iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter = max(0, x2 - x1) * max(0, y2 - y1)

    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])

    union = area1 + area2 - inter
    return inter / union if union > 0 else 0


def evaluate(model_path, data_yaml, split="test"):

    model = YOLO(model_path)

    metrics = model.val(data=data_yaml, split=split, plots=True)

    image_paths = glob.glob(f"dataset/{split}/images/*")

    all_ious = []

    for img_path in image_paths:

        img = cv2.imread(img_path)
        if img is None:
            continue

        h, w = img.shape[:2]

        label_path = img_path.replace("images", "labels").rsplit(".", 1)[0] + ".txt"

        gt_boxes = []

        if os.path.exists(label_path):
            with open(label_path) as f:
                for line in f:
                    cls, xc, yc, bw, bh = map(float, line.split())

                    x1 = (xc - bw/2) * w
                    y1 = (yc - bh/2) * h
                    x2 = (xc + bw/2) * w
                    y2 = (yc + bh/2) * h

                    gt_boxes.append([x1, y1, x2, y2])

        pred = model.predict(img_path, conf=0.25, verbose=False)[0]
        pred_boxes = pred.boxes.xyxy.cpu().numpy()

        for gt in gt_boxes:
            best = 0
            for p in pred_boxes:
                best = max(best, iou(gt, p))
            all_ious.append(best)

    mean_iou = np.mean(all_ious) if all_ious else 0

    summary = {
        "Precision": metrics.box.mp,
        "Recall": metrics.box.mr,
        "mAP50": metrics.box.map50,
        "mAP50-95": metrics.box.map,
        "Mean IoU": mean_iou
    }

    return summary, metrics

## 3. Training Pipeline (Freeze Comparison)

In [ ]:
from ultralytics import YOLO
import time
import pandas as pd

runs = [
    ("frozen_backbone", 11),
    ("partial_freeze_neck", 20),
    ("unfrozen", None)
]

results_test = []
results_val = []

for name, freeze in runs:

    print(f"\n===== Training {name} =====\n")

    model = YOLO("yolo26s.pt")

    start_time = time.time()

    train_args = {
        "data": "dataset.yaml",
        "epochs": 300,
        "patience": 50,
        "imgsz": 640,
        "batch": 16,

        "optimizer": "AdamW",
        "lr0": 0.0015,
        "lrf": 0.01,
        "warmup_epochs": 5,
        "warmup_bias_lr": 0.05,
        "weight_decay": 1e-4,

        "hsv_h": 0.015,
        "hsv_s": 0.7,
        "hsv_v": 0.4,
        "fliplr": 0.5,
        "flipud": 0.1,
        "scale": 0.5,
        "mosaic": 1.0,
        "copy_paste": 0.3,
        "degrees": 10.0,
        "translate": 0.1,

        "workers": 16,
        "amp": True,

        "project": "road_damage",
        "name": name,
        "exist_ok": True
    }

    if freeze is not None:
        train_args["freeze"] = freeze

    model.train(**train_args)

    duration = time.time() - start_time

    model_path = f"runs/detect/road_damage/{name}/weights/best.pt"

    # validation evaluation
    val_summary, _ = evaluate(model_path, "dataset.yaml", split="val")
    val_summary["Run"] = name
    results_val.append(val_summary)
    
    # test evaluation
    test_summary, _ = evaluate(model_path, "dataset.yaml", split="test")
    test_summary["Run"] = name
    test_summary["Time(min)"] = duration / 60
    results_test.append(test_summary)


===== Training frozen_backbone =====

Ultralytics 8.4.56 🚀 Python-3.9.25 torch-2.8.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.1, format=torchscript, fraction=1.0, freeze=11, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0015, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=frozen_backbone, nbs=64, nms=False, opset=None, optimize=False, optimiz

/home/achin/COS40007-Group/.venv/lib64/python3.9/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


val: Fast image access ✅ (ping: 1.7±2.0 ms, read: 53.5±53.4 MB/s, size: 85.3 KB)
val: Scanning /home/achin/COS40007-Group/jenny/dataset/val/labels.cache... 117 images, 18 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 117/117 6.1Mit/s 0.0s


/home/achin/COS40007-Group/.venv/lib64/python3.9/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 32 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


optimizer: AdamW(lr=0.0015, momentum=0.937) with parameter groups 114 weight(decay=0.0), 126 weight(decay=0.0001), 126 bias(decay=0.0)
Plotting labels to /home/achin/COS40007-Group/jenny/runs/detect/road_damage/frozen_backbone/labels.jpg... 
Image sizes 640 train, 640 val
Using 16 dataloader workers
Logging results to /home/achin/COS40007-Group/jenny/runs/detect/road_damage/frozen_backbone
Starting training for 300 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/300      2.67G      1.728       8.18    0.02375          2        640: 100% ━━━━━━━━━━━━ 35/35 4.9it/s 7.2s<0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 11.3it/s 0.4s.2s
                   all        117        165      0.152      0.125     0.0929     0.0351

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/300         3G       1.49      2.809    0.02142          2        

## 4. Compare Validation vs Test

In [ ]:
df_val = pd.DataFrame(results_val)
df_test = pd.DataFrame(results_test)

print("\n=== VALIDATION SET RESULTS ===")
print(df_val.sort_values(by="mAP50-95", ascending=False))

print("\n=== TEST SET RESULTS ===")
print(df_test.sort_values(by="mAP50-95", ascending=False))

## 5. Plot Loss + Metrics Curve

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

def plot_curves(run_name):

    df = pd.read_csv(f"runs/detect/road_damage/{run_name}/results.csv")

    plt.figure(figsize=(14,5))

    # Loss
    plt.subplot(1,2,1)
    for c in ["train/box_loss", "train/cls_loss", "train/dfl_loss"]:
        plt.plot(df["epoch"], df[c], label=c)
    plt.title(f"{run_name} Loss")
    plt.legend()

    # Metrics
    plt.subplot(1,2,2)
    for c in ["metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]:
        plt.plot(df["epoch"], df[c], label=c)
    plt.title(f"{run_name} Metrics")
    plt.legend()

    plt.tight_layout()
    plt.show()


for r in ["frozen_backbone", "partial_freeze_neck", "unfrozen"]:
    plot_curves(r)

## 6. Confusion Matrix Display

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

for r in ["frozen_backbone", "partial_freeze_neck", "unfrozen"]:

    img_path = f"runs/detect/road_damage/{r}/confusion_matrix_normalized.png"

    img = Image.open(img_path)

    plt.figure(figsize=(8,8))
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"{r} Confusion Matrix")
    plt.show()

## 7. Ground Truth vs Prediction with IoU

In [ ]:
from ultralytics import YOLO
import glob
import os
import cv2
import matplotlib.pyplot as plt

model = YOLO("runs/detect/road_damage/unfrozen/weights/best.pt")

for img_path in glob.glob("dataset/test/images/*"):

    img = cv2.imread(img_path)

    if img is None:
        continue

    h, w = img.shape[:2]

    pred = model.predict(
        img_path,
        conf=0.25,
        verbose=False
    )[0]

    pred_boxes = pred.boxes.xyxy.cpu().numpy()

    label_path = (
        img_path
        .replace("images", "labels")
        .rsplit(".", 1)[0] + ".txt"
    )

    gt_boxes = []

    if os.path.exists(label_path):

        with open(label_path) as f:

            for line in f:

                _, xc, yc, bw, bh = map(float, line.split())

                x1 = (xc - bw/2) * w
                y1 = (yc - bh/2) * h
                x2 = (xc + bw/2) * w
                y2 = (yc + bh/2) * h

                gt_boxes.append([x1, y1, x2, y2])

    # Draw predictions (RED)
    for p in pred_boxes:

        x1, y1, x2, y2 = map(int, p)

        cv2.rectangle(
            img,
            (x1, y1),
            (x2, y2),
            (0, 0, 255),
            2
        )

    # Draw ground truth and IoU (GREEN)
    for gt in gt_boxes:

        x1, y1, x2, y2 = map(int, gt)

        cv2.rectangle(
            img,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        best_iou = 0

        for pred_box in pred_boxes:

            current_iou = iou(gt, pred_box)

            if current_iou > best_iou:
                best_iou = current_iou

        cv2.putText(
            img,
            f"IoU={best_iou:.2f}",
            (x1, max(20, y1 - 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 0),
            2
        )

    # Legend
    cv2.putText(
        img,
        "Green = Ground Truth",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2
    )

    cv2.putText(
        img,
        "Red = Prediction",
        (10, 60),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 0, 255),
        2
    )

    plt.figure(figsize=(8, 8))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(os.path.basename(img_path))
    plt.axis("off")
    plt.show()

In [ ]:
from ultralytics import YOLO
import glob
import cv2
import os
import matplotlib.pyplot as plt

# =========================
# LOAD MODEL
# =========================
model = YOLO("runs/detect/road_damage/unfrozen/weights/best.pt")

class_names = model.names

print("Green = Ground Truth")
print("Red = Prediction")

# =========================
# VISUALIZE TEST IMAGES
# =========================
for img_path in glob.glob("dataset/test/images/*"):

    img = cv2.imread(img_path)

    if img is None:
        continue

    original_h, original_w = img.shape[:2]

    # =========================
    # ADD WHITE PADDING
    # =========================
    pad = 70

    img = cv2.copyMakeBorder(
        img,
        pad,
        pad,
        pad,
        pad,
        cv2.BORDER_CONSTANT,
        value=(255, 255, 255)
    )

    # =========================
    # PREDICTIONS
    # =========================
    pred = model.predict(
        img_path,
        conf=0.25,
        verbose=False
    )[0]

    pred_boxes = pred.boxes.xyxy.cpu().numpy()
    pred_classes = pred.boxes.cls.cpu().numpy()

    shifted_pred_boxes = []

    for box in pred_boxes:

        x1, y1, x2, y2 = box

        shifted_pred_boxes.append([
            x1 + pad,
            y1 + pad,
            x2 + pad,
            y2 + pad
        ])

    # =========================
    # LOAD GROUND TRUTH
    # =========================
    label_path = (
        img_path
        .replace("images", "labels")
        .rsplit(".", 1)[0] + ".txt"
    )

    gt_boxes = []
    gt_classes = []

    if os.path.exists(label_path):

        with open(label_path, "r") as f:

            for line in f:

                cls, xc, yc, bw, bh = map(float, line.split())

                x1 = (xc - bw / 2) * original_w
                y1 = (yc - bh / 2) * original_h
                x2 = (xc + bw / 2) * original_w
                y2 = (yc + bh / 2) * original_h

                gt_boxes.append([
                    x1 + pad,
                    y1 + pad,
                    x2 + pad,
                    y2 + pad
                ])

                gt_classes.append(int(cls))

    # =========================
    # DRAW PREDICTIONS (RED)
    # =========================
    for box, cls_id in zip(shifted_pred_boxes, pred_classes):

        x1, y1, x2, y2 = map(int, box)

        pred_name = class_names[int(cls_id)]

        cv2.rectangle(
            img,
            (x1, y1),
            (x2, y2),
            (0, 0, 255),
            2
        )

        cv2.putText(
            img,
            f"Pred: {pred_name}",
            (x1, max(20, y1 - 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (0, 0, 255),
            2
        )

    # =========================
    # DRAW GROUND TRUTH (GREEN)
    # =========================
    for gt_box, gt_cls in zip(gt_boxes, gt_classes):

        x1, y1, x2, y2 = map(int, gt_box)

        gt_name = class_names[gt_cls]

        cv2.rectangle(
            img,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        cv2.putText(
            img,
            f"GT: {gt_name}",
            (x1, y2 + 20),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (0, 255, 0),
            2
        )

        # =========================
        # FIND BEST MATCH
        # =========================
        best_iou = 0
        best_pred_class = "None"

        for pred_box, pred_cls in zip(shifted_pred_boxes, pred_classes):

            current_iou = iou(gt_box, pred_box)

            if current_iou > best_iou:

                best_iou = current_iou
                best_pred_class = class_names[int(pred_cls)]

        # =========================
        # LABEL AT UPPER RIGHT
        # =========================
        label_x = x2 + 15
        label_y = y1 - 5

        label1 = f"IoU: {best_iou:.2f}"
        label2 = f"Pred: {best_pred_class}"
        
        (tw1, th1), _ = cv2.getTextSize(
            label1,
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            2
        )
        
        (tw2, th2), _ = cv2.getTextSize(
            label2,
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            2
        )
        
        overlay = img.copy()
        
        cv2.rectangle(
            overlay,
            (label_x - 4, label_y - th1 - 4),
            (label_x + box_w, label_y + th2 + 15),
            (0, 0, 0),
            -1
        )
        
        alpha = 0.5  # transparency level
        
        cv2.addWeighted(
            overlay,
            alpha,
            img,
            1 - alpha,
            0,
            img
        )
        
        cv2.putText(
            img,
            label1,
            (label_x, label_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (255, 255, 0),
            1
        )
        
        cv2.putText(
            img,
            label2,
            (label_x, label_y + 18),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (255, 255, 0),
            1
        )

    # =========================
    # DISPLAY
    # =========================
    plt.figure(figsize=(10, 10))

    plt.imshow(
        cv2.cvtColor(
            img,
            cv2.COLOR_BGR2RGB
        )
    )

    plt.title(
        f"{os.path.basename(img_path)}\nGreen = Ground Truth | Red = Prediction"
    )

    plt.axis("off")

    plt.show()

In [ ]:
from ultralytics import YOLO
import glob
import cv2
import os
import matplotlib.pyplot as plt
import numpy as np

model = YOLO("runs/detect/road_damage/unfrozen/weights/best.pt")

def iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter = max(0, x2 - x1) * max(0, y2 - y1)
    a1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    a2 = (box2[2] - box2[0]) * (box2[3] - box2[1])

    return inter / (a1 + a2 - inter + 1e-6)


good_images = []

for img_path in glob.glob("dataset/test/images/*"):

    img = cv2.imread(img_path)
    if img is None:
        continue

    h, w = img.shape[:2]

    pred = model.predict(img_path, conf=0.25, verbose=False)[0]

    pred_boxes = pred.boxes.xyxy.cpu().numpy() if pred.boxes is not None else []
    pred_cls = pred.boxes.cls.cpu().numpy().astype(int) if pred.boxes is not None else []

    label_path = img_path.replace("images", "labels").rsplit(".", 1)[0] + ".txt"

    gt_boxes = []
    gt_cls = []

    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                c, xc, yc, bw, bh = map(float, line.split())

                x1 = (xc - bw / 2) * w
                y1 = (yc - bh / 2) * h
                x2 = (xc + bw / 2) * w
                y2 = (yc + bh / 2) * h

                gt_boxes.append([x1, y1, x2, y2])
                gt_cls.append(int(c))

    matched_preds = set()
    all_match = True

    for gt_box, gt_c in zip(gt_boxes, gt_cls):

        best_iou = 0
        best_j = -1

        for j, (pb, pc) in enumerate(zip(pred_boxes, pred_cls)):

            if pc != gt_c:
                continue

            score = iou(gt_box, pb)

            if score > best_iou:
                best_iou = score
                best_j = j

        if best_iou < 0.5 or best_j == -1:
            all_match = False
            break

        matched_preds.add(best_j)

    if len(matched_preds) != len(pred_boxes):
        all_match = False

    if not all_match:
        continue

    # draw GT + prediction
    for box, c in zip(gt_boxes, gt_cls):
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(img, f"GT: {model.names[c]}", (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    for box, c in zip(pred_boxes, pred_cls):
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(img, f"Pred: {model.names[c]}", (x1, y2 + 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

    good_images.append(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))


# =========================
# DISPLAY IN GRID (3 PER ROW)
# =========================
fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))

fig.suptitle("Good Prediction Examples", fontsize=2)

axes = axes.flatten() if len(good_images) > 1 else [axes]

for i, img in enumerate(good_images):
    ax = axes[i]
    ax.imshow(img)
    ax.axis("off")

# hide extra subplots
for j in range(len(good_images), len(axes)):
    axes[j].axis("off")

fig.tight_layout(rect=[0, 0, 1, 0.97])

plt.show()